In [1]:
import os
import time
import math
import torch
import torch.nn as nn
os.chdir('..')
from models.transformer import Transformer
from transformers import AutoTokenizer

/home/hussin/miniconda3/envs/caduceus_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def get_vram_mb():
    return torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0.0

In [3]:
device_name = torch.cuda.get_device_name(0)

In [4]:
device_name

'NVIDIA GeForce RTX 2080 Ti'

In [5]:
def detect_gpu_name():
    """Detects connected CUDA device name."""
    if not torch.cuda.is_available():
        return "CPU"

    return torch.cuda.get_device_name(0)


In [6]:
detect_gpu_name()

'NVIDIA GeForce RTX 2080 Ti'

In [7]:
GPU_PEAK_BANDWIDTH_GBS = { 
"NVIDIA GeForce RTX 2080 Ti": 448 
}

In [8]:
def generate_with_metrics(
    model,
    input_ids,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
    dtype_bytes=2  # 2 bytes for FP16 / BF16
):
    """
    Generates tokens while tracking TTFT, TPOT, Bytes Moved, Bandwidth (GB/s), and MBU (%).
    Decode phase is a single-token step (no loop).
    """
    model.eval()

    device = input_ids.device

    total_params = sum(p.numel() for p in model.parameters())
    weight_bytes = total_params * dtype_bytes
    _, prompt_len = input_ids.shape

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    start_event = torch.cuda.Event(enable_timing=True)
    prefill_event = torch.cuda.Event(enable_timing=True)
    decode_end_event = torch.cuda.Event(enable_timing=True)

    # --- PREFILL PHASE (first token) ---
    start_event.record()
    with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
        logits = model(input_ids)
        next_token_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
            next_token_logits[next_token_logits < v[:, [-1]]] = -float("Inf")

        probs = nn.functional.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

    prefill_event.record()
    curr_input_ids = torch.cat([input_ids, next_token], dim=1)

    # --- DECODE PHASE (single token, no loop) ---
    total_bytes_moved_decode = 0
    step_times_ms = []

    if max_new_tokens > 1:
        step_start = torch.cuda.Event(enable_timing=True)
        step_end = torch.cuda.Event(enable_timing=True)
        step_start.record()

        total_bytes_moved_decode = weight_bytes

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
            logits = model(curr_input_ids)
            next_token_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(next_token_logits, min(top_k, next_token_logits.size(-1)))
                next_token_logits[next_token_logits < v[:, [-1]]] = -float("Inf")

            probs = nn.functional.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        step_end.record()
        torch.cuda.synchronize()

        step_duration_ms = step_start.elapsed_time(step_end)
        step_times_ms.append(step_duration_ms)
        curr_input_ids = torch.cat([curr_input_ids, next_token], dim=1)

    decode_end_event.record()
    torch.cuda.synchronize()

    # Metrics
    ttft_ms = start_event.elapsed_time(prefill_event)
    total_decode_ms = prefill_event.elapsed_time(decode_end_event)
    avg_tpot_ms = step_times_ms[0] if step_times_ms else 0.0

    decode_tokens = 1 if max_new_tokens > 1 else 0
    throughput_tok_sec = (decode_tokens / total_decode_ms) * 1000.0 if total_decode_ms > 0 else 0.0
    total_decode_seconds = total_decode_ms / 1000.0
    achieved_bandwidth_gbs = (total_bytes_moved_decode / 1e9) / total_decode_seconds if total_decode_seconds > 0 else 0.0

    print(f"Achieved Bandwidth (GB/s): {achieved_bandwidth_gbs:.2f}")

    peak_bw = GPU_PEAK_BANDWIDTH_GBS.get(detect_gpu_name(), 0)
    mbu_percent = (achieved_bandwidth_gbs / peak_bw) * 100.0 if peak_bw > 0 else 0.0

    generated_tokens = curr_input_ids[0, prompt_len:].tolist()

    return {
        "generated_tokens": len(generated_tokens),
        "ttft_ms": round(ttft_ms, 2),
        "tpot_ms": round(avg_tpot_ms, 2),
        "throughput_tok_sec": round(throughput_tok_sec, 2),
        "total_decode_time_s": round(total_decode_seconds, 3),
        "total_gb_moved": round(total_bytes_moved_decode / 1e9, 4),
        "byte per token": round(total_bytes_moved_decode / decode_tokens, 2) if decode_tokens > 0 else 0.0,
        "achieved_bandwidth_gbs": round(achieved_bandwidth_gbs, 2),
        "mbu_percent": mbu_percent,
        "peak_vram_mb": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2)
    }, generated_tokens


### GEMV (sequence length of 1) for MHA basline model with torch.compile(mode = "default")

In [ ]:
checkpoint_path = "checkpoints/1_baseline_mha_sequential_res/model.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)

model = Transformer(**checkpoint["model_args"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# 1. Compile ONCE outside the benchmarking function
print("Compiling model...")
compiled_model = torch.compile(model, mode="default")

# 2. WARMUP PASS: Triggers initial compilation outside timing events
dummy_input = torch.tensor([[100, 200]], device=device)
with torch.no_grad(), torch.amp.autocast(device_type=device.type, dtype=torch.float16):
    _ = compiled_model(dummy_input)

# Wait for GPU compilation to complete completely
torch.cuda.synchronize()
print("Compilation complete. Starting benchmark...")

Compiling model...
Compilation complete. Starting benchmark...


In [ ]:
prompt = "Once upon a time in a deep forest, there lived a small"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# 1. Warmup run: Triggers compilation for Prefill [1, 12] AND Decode [1, 13]
_ = generate_with_metrics(compiled_model, input_ids, max_new_tokens=2)

# 2. Benchmarking run: ZERO compilation overhead
results, generated_tokens = generate_with_metrics(compiled_model, input_ids, max_new_tokens=2)

#decode the generated tokens to text
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(f"Generated Text: {generated_text}")
print(f"Metrics for Baseline Model:")
print(f"• TTFT (Prefill Latency): {results['ttft_ms']} ms")
print(f"• Generation Throughput: {results['throughput_tok_sec']} tok/sec")
print(f"• Peak Inference VRAM: {results['peak_vram_mb']} MB")
print(f"total decode seconds: {results['total_decode_time_s']} s")
print(f"byte per token: {results['byte per token']} bytes")
print(f"MBU (% of Peak Bandwidth): {results['mbu_percent']}%")


Achieved Bandwidth (GB/s): 15.18
Generated Text:  boy who
Metrics for Baseline Model:
• TTFT (Prefill Latency): 18009.14 ms
• Generation Throughput: 143.88 tok/sec
• Peak Inference VRAM: 552.95 MB
total decode seconds: 0.007 s
byte per token: 105516288.0 bytes
MBU (% of Peak Bandwidth): 3.3886577400252054%


### GEMV (sequence length of 1) for GQA basline model with torch.compile("default")

In [11]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()


In [12]:
checkpoint_path = "checkpoints/3_variant_b_gqa_2_kv_heads/model.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = Transformer(**checkpoint["model_args"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

tokenizer = AutoTokenizer.from_pretrained("gpt2")
# 1. Compile ONCE outside the benchmarking function
print("Compiling model...")
compiled_model = torch.compile(model, mode="default")

# 2. WARMUP PASS: Triggers initial compilation outside timing events
dummy_input = torch.tensor([[100, 200]], device=device)
with torch.no_grad(), torch.amp.autocast(device_type=device.type, dtype=torch.float16):
    _ = compiled_model(dummy_input)

# Wait for GPU compilation to complete completely
torch.cuda.synchronize()
print("Compilation complete. Starting benchmark...")

Compiling model...
Compilation complete. Starting benchmark...


In [ ]:
prompt = "Once upon a time in a deep forest, there lived a small"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# 1. Warmup run: Triggers compilation for Prefill [1, 12] AND Decode [1, 13]
_ = generate_with_metrics(compiled_model, input_ids, max_new_tokens=2)

# 2. Benchmarking run: ZERO compilation overhead
results, generated_tokens = generate_with_metrics(compiled_model, input_ids, max_new_tokens=2)

#decode the generated tokens to text
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(f"Generated Text: {generated_text}")
print(f"Metrics for GQA Model:")
print(f"• TTFT (Prefill Latency): {results['ttft_ms']} ms")
print(f"• Generation Throughput: {results['throughput_tok_sec']} tok/sec")
print(f"• Peak Inference VRAM: {results['peak_vram_mb']} MB")
print(f"byte per token: {results['byte per token']} bytes")
print(f"MBU (% of Peak Bandwidth): {results['mbu_percent']}%")

Achieved Bandwidth (GB/s): 22.21
Generated Text:  boy named
Metrics for GQA Model:
• TTFT (Prefill Latency): 19177.9 ms
• Generation Throughput: 215.29 tok/sec
• Peak Inference VRAM: 746.67 MB
byte per token: 103156992.0 bytes
MBU (% of Peak Bandwidth): 4.9572937983721355%
